### Planned changes so far:
- Coordinate aware embeddings
- see Email in research log.docx
    - i mean the goal is to construct a better gpt model
        - but also a better training in general thats why i will run tests with a direct comparison between the profs and  my model (his training and eval) and then also change metricies like the Learningratescheduler and see if i can achieve higher performance

In [ ]:
import torch
import torch.nn as nn
import math
from spike_gpt import train

In [ ]:
class XYZAwareLearnedSessionAdapter(nn.Module):
    """Coordinate-aware spike embeddings + per-task heads.

    - Shared MLPs (coord_mlp_in / coord_mlp_out) dynamically generate 
      w_in (N, d) and w_out (N, d) from 3D unit coordinates.
    - Per-session b_out (N,) bias initialized to mean firing rate.
    - Per-task embedding (d,) to condition the model on the current task.
    """
    def __init__(self, n_units_per_session, task_specs, d, device,
                 mean_rates=None, session_coords=None):
        super().__init__()
        self.d = d
        self.session_coords = [
            torch.tensor(c, dtype=torch.float32, device=device)
            for c in session_coords
        ]
        
        from gradient_normalizer import GradientNormalizer
        self.grad_normalizers = nn.ModuleDict({
            spec.name: GradientNormalizer(normalizer_shape=[1], scale=1.0)
            for spec in task_specs
        })
        self.task_embeds = nn.ParameterDict({
            spec.name: nn.Parameter(torch.randn(d, device=device) * 0.02)
            for spec in task_specs
        })
        
        # coordnate aware mlp
        self.coord_mlp_in = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d),
        )
        
        self.coord_mlp_out = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d),
        )

        # b_out initialized so that relu(0 + b_out) = mean firing rate per neuron
        self._b_out = nn.ParameterList([
            nn.Parameter(torch.from_numpy(mean_rates[i]).float().to(device))
            for i in range(len(n_units_per_session))
        ])
        # Per-task prediction heads
        task_heads = {}
        for spec in task_specs:
            if isinstance(spec, RegressionTask):
                task_heads[spec.name] = nn.Parameter(
                    torch.randn(spec.d_out, d, device=device) * (1.0 / d**0.5))
            elif isinstance(spec, ClassificationTask):
                task_heads[spec.name] = nn.Parameter(
                    torch.randn(spec.n_classes, d, device=device) * (1.0 / d**0.5))
        self.task_heads = nn.ParameterDict(task_heads)

    def w_in(self, session, visible_idx=None):
        coords = self.session_coords[session]
        if visible_idx is not None:
            coords = coords[visible_idx]
        return self.coord_mlp_in(coords)

    def w_out(self, session):
        return self.coord_mlp_out(self.session_coords[session])

    def b_out(self, session):
        return self._b_out[session]

In [ ]:
class CastedLinear(nn.Linear):
    """Linear layer that casts weights to input dtype (keeps master weights in float32)."""
    def forward(self, x):
        return F.linear(x, self.weight.to(x.dtype))

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert config.n_embd % config.n_head  == 0, "n_embd must be divisible by num_head"
        
        # Initialize dimensions
        self.num_head = config.n_head # Number of attention heads
        self.d_k = config.n_embd // config.n_head # Dimension of each head's key, query, and value
        
        # Linear layers for transforming inputs
        self.W_q = CastedLinear(config.n_embd, config.n_embd, bias=False) # Query transformation
        self.W_k = CastedLinear(config.n_embd, config.n_embd, bias=False) # Key transformation
        self.W_v = CastedLinear(config.n_embd, config.n_embd, bias=False) # Value transformation
        self.W_o = CastedLinear(config.n_embd, config.n_embd, bias=False) # Output transformation
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            if hasattr(mask, "to_dense"):
                # to_dense() converts PyTorch-Object to a readable (1,1,Seq, Seq) bool mask
                dense_mask = mask.to_dense()
                attn_scores = attn_scores.masked_fill(~dense_mask, -1e9)
            else:
                attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_head, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc1 = CastedLinear(config.n_embd, 4 * config.n_embd, bias=False)
        self.fc2 = CastedLinear(4 * config.n_embd, config.n_embd, bias=False)
        self.fc2.weight.data.zero_()

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)).square())

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal Positional Encoding
    uses passed positions"""
    def __init__(self, config):
        super().__init__()
        self.d_model = config.n_embd
        
        div_term = torch.exp(torch.arange(0, self.d_model, 2).float() * -(math.log(10000.0) / self.d_model))
        self.register_buffer('div_term', div_term)
        
    def forward(self, x, positions):
        if positions.dim() == 1:
            positions = positions.unsqueeze(0).expand(x.size(0), -1)
        
        pe = torch.zeros(x.size(0), x.size(1), self.d_model, device=x.device, dtype=x.dtype)
        pos_scaled = positions.unsqueeze(-1).float() * self.div_term
        
        pe[..., 0::2] = torch.sin(pos_scaled)
        pe[..., 1::2] = torch.cos(pos_scaled)

        return x + pe
        
        

Below is the gpt block that replaces encoder and decoder from traditional transformer

In [ ]:
class GPTBlock(nn.Module):
    """Der Decoder-Block: Verknüpft Attention und MLP."""
    def __init__(self, config):
        super().__init__()
        self.self_attn = MultiHeadAttention(config)
        self.feed_forward = PositionWiseFeedForward(config)
        
        # DataCamp nutzt LayerNorm (dein Prof nutzte RMSNorm)
        self.norm1 = nn.LayerNorm(config.n_embd)
        self.norm2 = nn.LayerNorm(config.n_embd)

    def forward(self, x, block_mask, positions=None):
        # Attention (mit Pre-Norm für bessere Stabilität im Training)

        # positions wird ignoriert weil Die Positionen werden 
        # ein einziges Mal ganz am Anfang (in der SpikeGPT-Klasse) 
        # auf den Input x addiert
        _ = positions
        
        x = x + self.self_attn(self.norm1(x), block_mask)
        x = x + self.feed_forward(self.norm2(x))
        return x

In [ ]:
class SpikeGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.sos_embed = nn.Parameter(torch.randn(config.n_embd) * 0.02)
        self.positional_encoding = PositionalEncoding(config)
        self.blocks = nn.ModuleList([GPTBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)

    def forward(self, x, block_mask, positions=None):
        if positions is None:
            positions = torch.arange(x.size(1), device=x.device, dtype=torch.float)

        x = self.positional_encoding(x, positions)

        for block in self.blocks:
            x = block(x, block_mask, positions)

        return self.final_norm(x)

# Training the Transformer model
### Sample Data preperation

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    model="tiny",
    seq_len=None,
    dt=0.01,
    datasets="allen",
    n_max_sessions=20,
    use_all_sessions=False,
    n_epochs=100,
    lr=3e-4,
    mask_range=(0.1, 0.5),
    warmup_steps=1000,
    log_every=50,
    batch_size=32,
    n_tasks_per_step=None,
    no_grad_norm=True,
    n_sos_masks=128,
    n_hops=10,
    eval_only=None,
    cache_dir="/scratch/neuro-ai/cache",
    data_root="/share/datasets/neuro-ai",
)

if args.seq_len is None:
    args.seq_len = 16 * CONFIGS[args.model].window_size


### Training the Model

In [ ]:
train(args)